# old-imagery: a tour

`old-imagery` fetches historical aerial imagery from **Google Earth** and the
**Esri World Imagery Wayback** archive. Areas of interest go in as shapely
geometries; availability comes back as a GeoDataFrame, and imagery comes back
as an open rasterio dataset.

This notebook walks the whole public API:

1. `availability` — which capture dates exist over an area
2. `download` — the pixels for one of them, and what the raster tags tell you
3. the same two calls against Esri Wayback
4. `esri_mosaic_as_of` — what a published Esri basemap *displayed* on a date
5. saving, guardrails and caching

> **This library retrieves imagery. It does not license it.** Everything fetched
> below stays the copyright of Google, Esri or their imagery providers, and
> their terms of service govern what you may do with it — read the notice at the
> top of the [README](../README.md) before using any of it. The areas here are
> deliberately tiny.

The committed copy of this notebook has its outputs stripped, so it carries no
redistributed imagery. Run it yourself to see the pictures.

## Setup

Beyond the library this needs `matplotlib` for the plots:
`pip install "old-imagery[examples]"`.

In [ ]:
%matplotlib inline

import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
from rasterio.plot import show
from rasterio.shutil import copy as rio_copy
from shapely.geometry import Point, box

import old_imagery

print("old-imagery", old_imagery.__version__)
print("cache dir  ", old_imagery.DEFAULT_CACHE_DIR)
print("zoom limits", old_imagery.MAX_IMAGERY_ZOOM)

Every `aoi` argument is a shapely `Polygon`, `MultiPolygon` or
`GeometryCollection` enclosing some area, interpreted as longitude/latitude in
**EPSG:4326**. Shapely geometries do not carry a CRS, so reproject before
calling the library.

Two areas are used below: a few blocks of the San Francisco Embarcadero, and a
small area across the bay that happens to straddle a seam between two Esri
capture dates.

In [ ]:
EMBARCADERO = box(-122.4020, 37.7900, -122.3880, 37.7990)
SEAM = box(-122.3484, 37.89444, -122.3460, 37.89684)

## 1. Which capture dates exist here?

`availability` returns one row per capture date, newest first, in EPSG:4326.
`zoom` sets how finely date boundaries are traced — 15–17 is usually the right
trade-off, since the date *list* barely changes above that, only the precision
of the polygons.

In [ ]:
dates = old_imagery.availability(EMBARCADERO, zoom=17)

print(f"{len(dates)} capture dates, {dates['date'].min()} to {dates['date'].max()}")
dates[["date", "coverage", "complete", "providers"]].head()

`coverage` is the fraction of the AOI's **area** covered by imagery from that
date, computed as a planar ratio in EPSG:3857. It means the same thing for both
providers and does not depend on `zoom`. `complete` is just `coverage == 1.0`.

`gdf.attrs` records how the answer was obtained. It reports what ran; it is not
selectable.

In [ ]:
print(dates.attrs)
print(f"{dates['complete'].sum()} of {len(dates)} dates cover the whole AOI")

Date bounds are inclusive, and narrow the search:

In [ ]:
nineties = old_imagery.availability(
    EMBARCADERO, zoom=17, min_date="1990-01-01", max_date="1999-12-31"
)
nineties[["date", "coverage", "complete"]]

The `date` column — and every date this package reports — is the date the
imagery was **captured**, never a provider's publication or release date.
Publication dates are named separately: `release_date` in the Wayback catalogue
and `as_of` in `esri_mosaic_as_of`, in section 4.

## 2. Getting the pixels

`download` returns an open, in-memory 3-band uint8 RGB `rasterio.DatasetReader`
covering the AOI's bounding box, snapped to tile pixels. It is a context
manager, and the data lives in memory — so read or plot inside the `with`.

In [ ]:
oldest = dates["date"].iloc[-1]

with old_imagery.download(EMBARCADERO, zoom=17, date=oldest, date_match="exact") as src:
    fig, ax = plt.subplots(figsize=(7, 5))
    show(src, ax=ax, title=f"Embarcadero, {oldest}")
    ax.set_axis_off()
    tags = src.tags()
    print(src.crs, src.shape, src.count, src.dtypes[0])

For unchanged provider image payloads rather than a mosaic, `download_tiles`
accepts a point or polygon and returns complete native tiles. It is strict: any
missing selected tile raises instead of returning a partial list. Payloads stay in
memory; pass `cache_dir=None` when even HTTP caching must not write to disk.

In [ ]:
raw_tile = old_imagery.download_tiles(
    EMBARCADERO.centroid, zoom=17, date=oldest, date_match="exact"
)[0]
print(raw_tile.image_format, len(raw_tile.content), raw_tile.tile_scheme)
print(raw_tile.capture_date_at_center, raw_tile.bounds_wgs84)

The CRS is each provider's **native tile grid** — `EPSG:4326` for Google,
`EPSG:3857` for Esri — so nothing is resampled on the way out. Use
`rasterio.warp` if you need something else.

Every download describes itself in its tags:

In [ ]:
tags

### `date_match`, and why a mosaic can mix dates

Tiles are matched to the target date one at a time, so `date_match` controls
the per-tile fallback when a tile has nothing on that date: `"closest"` (the
default), `"exact"`, `"before"` or `"after"`. Because the fallback is per tile,
one mosaic can end up stitched from several capture dates — which is what the
`dates` tag is for. It lists every date that actually made it into the pixels:
one here, since the whole AOI fell back together, but two for the Esri release
in section 4.

In [ ]:
with old_imagery.download(EMBARCADERO, zoom=17, date="1975-01-01") as src:
    loose = src.tags()

print("asked for  ", loose["target_date"], f"({loose['date_match']})")
print("got        ", loose["dates"])
print("tiles      ", loose["tiles_total"], "total,", loose["tiles_missing"], "missing")

Tiles with no imagery at all are left black and excluded by the dataset mask,
so use `src.dataset_mask()` or `src.read(masked=True)` to ignore gaps rather
than trusting the black pixels.

### Then and now

The same frame at both ends of the archive:

In [ ]:
newest = dates["date"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, when in zip(axes, [oldest, newest], strict=True):
    with old_imagery.download(EMBARCADERO, zoom=17, date=when, date_match="exact") as src:
        show(src, ax=ax, title=str(when))
    ax.set_axis_off()

## 3. The same two calls against Esri Wayback

`provider="esri"` switches both `availability` and `download` to the Esri World
Imagery Wayback archive. Esri publishes real capture footprints, so date
boundaries follow actual imagery seams rather than tile edges, and the rows
carry source provenance Google does not provide.

Esri is slow: Wayback exposes no bulk per-tile date query, so this issues a few
hundred metadata queries and takes tens of seconds on a cold cache. Keep the
area small.

In [ ]:
esri_dates = old_imagery.availability(SEAM, zoom=17, provider="esri")

print(esri_dates.attrs)
esri_dates[["date", "coverage", "source_providers", "source_resolutions_m"]].head()

## 4. What did the basemap *display* on a given day?

`availability` asks which capture dates exist. `esri_mosaic_as_of` asks a
different question: if you had looked at the Esri basemap on this date, what
would you have been seeing, and how old is each part of it?

A Wayback *release* is a mosaic stitched from imagery flown across many years,
so "what it displays" is its internal seam map. `as_of` accepts either a
**publication** date (selecting the latest release on or before it) or an exact
stable ID from `esri_wayback_releases()`.

In [ ]:
releases = old_imagery.esri_wayback_releases()
print(releases.head())

seams = old_imagery.esri_mosaic_as_of(SEAM, zoom=18, as_of="2020-06-01")

print(seams.attrs["release_title"], "| published", seams.attrs["release_date"])
seams[["zoom", "date", "area_fraction", "source_provider", "source_resolution_m"]]

Those are real footprint boundaries, not tile edges — this AOI is split between
two flights eleven days apart:

In [ ]:
ax = seams.assign(capture=seams["date"].astype(str)).plot(
    column="capture", categorical=True, legend=True, edgecolor="white", figsize=(6, 5)
)
ax.set_title(f"What {seams.attrs['release_title']} displayed")
ax.set_axis_off()

### Zoom is an axis here, not just a resolution knob

Esri composes the mosaic per scale and publishes metadata per scale, so the same
ground in the same release can show a *different capture date* at different
zooms. Pass several zooms to see that. Cost scales with the number of capture
footprints rather than tiles, which is why the guard here is `max_footprints`
and there is no `max_tiles`.

In [ ]:
by_zoom = old_imagery.esri_mosaic_as_of(SEAM, [13, 16, 19], "2020-06-01")
by_zoom.groupby("zoom")["date"].agg(lambda dates: sorted(set(dates)))

### Round-tripping a seam map to pixels

`download` accepts the `release_id` this reports, so the map and the imagery
always refer to the same release. This is the only release selector on
`download`; it requires `provider="esri"` and no `date`.

In [ ]:
release_id = seams.attrs["release_id"]

with old_imagery.download(
    SEAM, zoom=18, provider="esri", esri_wayback_release_id=release_id
) as src:
    fig, ax = plt.subplots(figsize=(6, 5))
    show(src, ax=ax, title=release_id)
    ax.set_axis_off()
    by_release = src.tags()

print(by_release["selection_mode"], "|", by_release["dates"])
print(by_release["tiles_missing"], "of", by_release["tiles_total"], "tiles missing")

A release is a mosaic of many capture dates, so no single `date=` reproduces
one. Asking for one capture date masks out everything flown on any other:

In [ ]:
with old_imagery.download(
    SEAM, zoom=18, provider="esri", date=seams["date"].iloc[0], date_match="exact"
) as src:
    by_date = src.tags()

print(f"by release: {by_release['tiles_missing']:>3} missing, dates {by_release['dates']}")
print(f"by date:    {by_date['tiles_missing']:>3} missing, dates {by_date['dates']}")

## 5. Saving, guardrails and caching

`download` hands back an in-memory dataset. To put it on disk without touching
its pixels or georeferencing, copy it straight out with rasterio:

In [ ]:
out = Path(tempfile.mkdtemp()) / "embarcadero.tif"  # your own path here

with old_imagery.download(EMBARCADERO, zoom=17, date=oldest, date_match="exact") as src:
    rio_copy(src, out, driver="GTiff")

print(out, f"{out.stat().st_size / 1e6:.1f} MB")

### Guardrails

The failure modes are ordinary `ValueError`s with messages that say what to do,
raised before anything hits the network. Network failures raise
`old_imagery.RequestFailed` (and `NotFound`, a subclass of it).

In [ ]:
def report(call):
    """Run a call that is expected to fail, and print the complaint."""
    try:
        call()
    except (ValueError, old_imagery.RequestFailed) as exc:
        print(f"{type(exc).__name__}: {exc}\n")


report(lambda: old_imagery.availability(Point(-122.4, 37.79), zoom=17))
report(lambda: old_imagery.availability(box(-123, 37, -122, 38), zoom=17))
report(lambda: old_imagery.availability(EMBARCADERO, zoom=22))
report(lambda: old_imagery.download(EMBARCADERO, zoom=17, date="1900-01-01", date_match="exact"))

### Caching

Responses are cached on disk in the location your platform expects, which is why
re-running the cells above is fast. Keyhole assets are addressed by epoch and
therefore immutable, so entries never go stale; only Google's dbRoot and Esri's
capabilities document are re-fetched, daily and weekly.

Override the location with `$OLD_IMAGERY_CACHE_DIR` before importing the
package, or per call with `cache_dir=`; pass `cache_dir=None` to disable it.
There is no size limit or eviction policy, so clear the directory yourself if a
long-running workflow lets it grow.

In [ ]:
cache = Path(old_imagery.DEFAULT_CACHE_DIR)
if cache.exists():
    files = list(cache.rglob("*"))
    size = sum(f.stat().st_size for f in files if f.is_file())
    print(f"{cache}\n{len(files):,} entries, {size / 1e6:.1f} MB")

### There is no concurrency knob

Every parallel section here is network-bound, so the thread pool is sized by
what the *service* tolerates and by how much work there is — never more threads
than tasks, and never more than the per-provider cap (16 for Google, 10 for
Esri). The caps are fixed rather than adaptive: finding a service's limit means
exceeding it, and Google's terms prohibit bulk feeds.

---

That is the whole public surface: `availability`, `download`,
`esri_wayback_releases`, `esri_mosaic_as_of`, the `MAX_IMAGERY_ZOOM` and
`DEFAULT_CACHE_DIR` constants,
and the `RequestFailed` / `NotFound` exceptions. Everything else lives in
underscore modules and tracks Google's and Esri's wire formats, so it changes
when they do. See the [README](../README.md) for the full reference.